In [1]:
from langchain_community.retrievers import WikipediaRetriever

### Wikipedia Retriever:

In [2]:
retriever=WikipediaRetriever(
    top_k_results=2,
    lang="en"
)

In [5]:
query="Tell me about Rohit Sharma and 2024 cricket world cup"
doc=retriever.invoke(query)

In [6]:
doc

[Document(metadata={'title': 'The Kapil Sharma Show', 'summary': "The Kapil Sharma Show, also known as TKSS, is an Indian Hindi language stand-up comedy and talk show broadcast by Sony Entertainment Television. Hosted by Kapil Sharma, the first season of the show premiered on 23 April 2016. The series revolved around Sharma and his neighbours in the Shantivan Non Co-operative Housing Society. The filming of the show took place at Film City situated in Goregaon East, Mumbai. The first season of the show was produced by Sharma's banner K9 Productions in association with Frames Productions while the second and third season were jointly produced by Salman Khan Television and Banijay Asia with K9 Productions and TEAM (Triyambh Entertainment and Media) as the creative producers. The show's fifth season was launched in September 2022 in which Archana Puran Singh reprised her role as the guest judge. As per the reports, the team also saw some new actors joining the star cast.", 'source': 'http

In [10]:
for i, content in enumerate(doc):
    print(f"Result--{i+1}")
    print("content:\n",content.page_content)
    print("*"*100)

Result--1
content:
 The Kapil Sharma Show, also known as TKSS, is an Indian Hindi language stand-up comedy and talk show broadcast by Sony Entertainment Television. Hosted by Kapil Sharma, the first season of the show premiered on 23 April 2016. The series revolved around Sharma and his neighbours in the Shantivan Non Co-operative Housing Society. The filming of the show took place at Film City situated in Goregaon East, Mumbai. The first season of the show was produced by Sharma's banner K9 Productions in association with Frames Productions while the second and third season were jointly produced by Salman Khan Television and Banijay Asia with K9 Productions and TEAM (Triyambh Entertainment and Media) as the creative producers. The show's fifth season was launched in September 2022 in which Archana Puran Singh reprised her role as the guest judge. As per the reports, the team also saw some new actors joining the star cast.


== Premise ==
The format is largely identical  of Sharma's fo

#### vector store retriever

In [11]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.documents import Document

In [12]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [13]:
## initialized embedding model
model=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

## creating vector database

vector_store=Chroma.from_documents(
    embedding=model,
    documents=documents,
    collection_name="my_collection"

)

In [14]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['c7974e1a-74f2-43a6-925c-6a7f20295660',
  'e7b49cde-b401-4110-abe6-5db4c989242e',
  '3ffe0466-8542-4a54-a165-1022293b286f',
  '4af959ac-ee5e-4d70-8e36-160cc7df1f91'],
 'embeddings': array([[-0.01342658, -0.01894311,  0.05097282, ...,  0.01465148,
          0.08020388,  0.01772903],
        [-0.02399961,  0.01149502, -0.04277549, ..., -0.00242297,
          0.0272245 , -0.01669936],
        [-0.00697725, -0.02583647,  0.00157223, ...,  0.05452187,
          0.04645807, -0.01169451],
        [ 0.00088637, -0.11448762, -0.00186407, ...,  0.0291839 ,
          0.04197901,  0.04242006]], shape=(4, 384)),
 'documents': ['LangChain helps developers build LLM applications easily.',
  'Chroma is a vector database optimized for LLM-based search.',
  'Embeddings convert text into high-dimensional vectors.',
  'OpenAI provides powerful embedding models.'],
 'uris': None,
 'included': ['embeddings', 'documents', 'metadatas'],
 'data': None,
 'metadatas': [None, None, None, None]}

##### Convert vector store as retriever

In [19]:
retriever=vector_store.as_retriever(search_kwargs={'k':2})

In [20]:
query="what is chroma used for"
result=retriever.invoke(query)

for i, doc in enumerate(result):
    print(f"result--->{i+1}")
    print(f"content-->\n{doc.page_content}")

result--->1
content-->
Chroma is a vector database optimized for LLM-based search.
result--->2
content-->
LangChain helps developers build LLM applications easily.


In [21]:
result1=vector_store.similarity_search(query=query,k=2)
for i, doc in enumerate(result1):
    print(f"result--->{i+1}")
    print(f"content-->\n{doc.page_content}")


result--->1
content-->
Chroma is a vector database optimized for LLM-based search.
result--->2
content-->
LangChain helps developers build LLM applications easily.


### MMR

In [22]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [23]:
from langchain_community.vectorstores import FAISS

In [24]:
vectorstore=FAISS.from_documents(
    embedding=model,
    documents=docs
    
)

In [29]:
## Enable MMR in retriever
retriever=vectorstore.as_retriever(search_type='mmr',search_kwargs={'k':3,'lambda_mult':0.60})

In [30]:
query

'what is chroma used for'

In [31]:
result=retriever.invoke(query)

for i, doc in enumerate(result):
    print(f"result--->{i+1}")
    print(f"content-->\n{doc.page_content}")


result--->1
content-->
Chroma is used to store and search document embeddings.
result--->2
content-->
LangChain supports Chroma, FAISS, Pinecone, and more.
result--->3
content-->
MMR helps you get diverse results when doing similarity search.


### Multi query retriever

In [44]:
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.documents import Document
from langchain.retrievers.multi_query import MultiQueryRetriever
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS,Chroma
import os

In [45]:
load_dotenv()

True

In [35]:
all_docs=[
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"})
]

In [ ]:
#embed_model create
embed_model=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

#llm model
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    #huggingfacehub_api_token="hf_tJPxeBDKQpWvwWwXIy........."  #
    
)

llm_model=ChatHuggingFace(llm=llm)

In [52]:
res=llm_model.invoke("Give me two line description about rohit sharma")
print(res.content)

Rohit Sharma is an Indian international cricketer renowned for his explosive batting and record‑breaking 100s in limited‑overs cricket.  
The left‑handed opener has led India to numerous victories and is one of the few players to score three double centuries in ODIs.


In [37]:
vector_str=FAISS.from_documents(
    embedding=embed_model,
    documents=all_docs
)

In [65]:
## creating retriever
mul_qu_retriever=MultiQueryRetriever.from_llm(
    llm=llm_model,
    retriever=vector_str.as_retriever(search_kwargs={'k': 2}) 
)

In [66]:
query="How to improve energy levels and maintain balance?"
result=mul_qu_retriever.invoke(query)

for i, doc in enumerate(result):
    print(f"result--->{i+1}")
    print(f"content-->\n{doc.page_content}")

result--->1
content-->
Drinking sufficient water throughout the day helps maintain metabolism and energy.
result--->2
content-->
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
result--->3
content-->
The solar energy system in modern homes helps balance electricity demand.
result--->4
content-->
Consuming leafy greens and fruits helps detox the body and improve longevity.


In [67]:
len(result)

4

### ContextualCompressionRetriever

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [69]:
set_docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [70]:
vctr_str=FAISS.from_documents(
    embedding=embed_model,
    documents=set_docs
)

In [71]:
# base retriever
base_retriever=vctr_str.as_retriever(search_kwargs={'k':5})

In [ ]:
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    #huggingfacehub_api_token="hf_VoInNGsjhLPMar********"   
    
)

llm_com=ChatHuggingFace(llm=llm)

In [79]:
base_compressor=LLMChainExtractor.from_llm(llm_com)

In [80]:
compressor_retriever=ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=base_compressor
)

In [81]:
query="What is photosynthesis? "
output=compressor_retriever.invoke(query)
for i, doc in enumerate(output):
    print(f"result--->{i+1}")
    print(f"content-->\n{doc.page_content}")

result--->1
content-->
Photosynthesis is the process by which green plants convert sunlight into energy.
result--->2
content-->
The chlorophyll in plant cells captures sunlight during photosynthesis.
result--->3
content-->
Photosynthesis does not occur in animal cells.


In [76]:
len(output)

3